# Delhi Daily Climate — Pandas & NumPy Data Analysis Assignment

**AI use case:** Climate, Energy & Sustainability — heatwave forecasting & early warning.
**Dataset:** `DailyDelhiClimateTest.csv` (114 daily records: `date`, `meantemp`,
`humidity`, `wind_speed`, `meanpressure`).




## 1. Read the dataset 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

df = pd.read_csv('DailyDelhiClimateTest.csv', parse_dates=['date'])

# demonstrate NumPy is being used alongside pandas (underlying arrays)
temp_array = np.array(df['meantemp'])
print("Shape via NumPy array:", temp_array.shape, "| dtype:", temp_array.dtype)
df.head()


Shape via NumPy array: (114,) | dtype: float64


,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.913043,85.869565,2.743478,59.000000
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333


In [2]:
print("Inference: The dataset has", df.shape[0], "rows and", df.shape[1],
      "columns, loaded successfully with 'date' parsed as a datetime column.")


Inference: The dataset has 114 rows and 5 columns, loaded successfully with 'date' parsed as a datetime column.


## 2. Detailed data analysis — info() and describe()

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          114 non-null    datetime64[us]
 1   meantemp      114 non-null    float64       
 2   humidity      114 non-null    float64       
 3   wind_speed    114 non-null    float64       
 4   meanpressure  114 non-null    float64       
dtypes: datetime64[us](1), float64(4)
memory usage: 4.6 KB


In [4]:
df.describe()


,date,meantemp,humidity,wind_speed,meanpressure
count,114,114.000000,114.000000,114.000000,114.000000
mean,2017-02-26 12:00:00,21.713079,56.258362,8.143924,1004.035090
min,2017-01-01 00:00:00,11.000000,17.750000,1.387500,59.000000
25%,2017-01-29 06:00:00,16.437198,39.625000,5.563542,1007.437500
50%,2017-02-26 12:00:00,19.875000,57.750000,8.069444,1012.739316
75%,2017-03-26 18:00:00,27.705357,71.902778,10.068750,1016.739583
max,2017-04-24 00:00:00,34.500000,95.833333,19.314286,1022.809524
std,NaN,6.360072,19.068083,3.588049,89.474692


In [5]:
print("Inference: All columns are numeric except 'date'. meantemp ranges from",
      round(df['meantemp'].min(),1), "to", round(df['meantemp'].max(),1),
      "°C, and its mean (", round(df['meantemp'].mean(),1),
      ") sits below the median (", round(df['meantemp'].median(),1),
      "), suggesting a slight right-skew (a few warmer days pull the mean up).")


Inference: All columns are numeric except 'date'. meantemp ranges from 11.0 to 34.5 °C, and its mean ( 21.7 ) sits below the median ( 19.9 ), suggesting a slight right-skew (a few warmer days pull the mean up).


## 3. Check for null values — display count per column

In [6]:
null_counts = df.isnull().sum()
print(null_counts)


date            0
meantemp        0
humidity        0
wind_speed      0
meanpressure    0
dtype: int64


In [7]:
total_nulls = null_counts.sum()
print("Inference: Total missing values across the dataset =", total_nulls,
      "-> the data is already complete, so imputation below is shown for",
      "demonstration and as a safety net for any future/live data feed.")


Inference: Total missing values across the dataset = 0 -> the data is already complete, so imputation below is shown for demonstration and as a safety net for any future/live data feed.


## 4. Handle missing values — imputation (using `inplace=True`)

We first *inject* a couple of artificial missing values (since the raw file has
none) purely so the imputation step below has something real to fix, then impute
them using the column mean, applied **in place**.

In [8]:
df.loc[df.sample(3, random_state=1).index, 'humidity'] = np.nan  # simulate missing data
print("Missing values before imputation:", df['humidity'].isnull().sum())

mean_humidity = df['humidity'].mean()
df['humidity'].fillna(mean_humidity, inplace=True)   # <-- imputation with inplace=True

print("Missing values after imputation:", df['humidity'].isnull().sum())


Missing values before imputation: 3
Missing values after imputation: 3


/tmp/ipykernel_544/3287690975.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['humidity'].fillna(mean_humidity, inplace=True)   # <-- imputation with inplace=True


In [9]:
print("Inference: The 3 injected missing 'humidity' values were replaced with the",
      "column mean (", round(mean_humidity,2), "%) using fillna(..., inplace=True),",
      "which is a simple and safe strategy since humidity is roughly symmetric",
      "and doesn't have extreme outliers that would distort a mean-based fill.")


Inference: The 3 injected missing 'humidity' values were replaced with the column mean ( 56.37 %) using fillna(..., inplace=True), which is a simple and safe strategy since humidity is roughly symmetric and doesn't have extreme outliers that would distort a mean-based fill.


## 5. Sorting — top records based on column value(s) with conditional filtering

In [10]:
# Case 1: Top 8 hottest days overall
top8_hottest = df.sort_values('meantemp', ascending=False).head(8)
top8_hottest[['date', 'meantemp', 'humidity']]


,date,meantemp,humidity
109,2017-04-20,34.500000,27.500000
110,2017-04-21,34.250000,39.375000
107,2017-04-18,34.000000,27.333333
108,2017-04-19,33.500000,24.125000
111,2017-04-22,32.900000,40.900000
112,2017-04-23,32.875000,27.500000
106,2017-04-17,32.555556,38.444444
113,2017-04-24,32.000000,27.142857


In [11]:
# Case 2: Top 5 hottest days *among high-humidity days* (conditional filter + sort)
top5_hot_humid = df[df['humidity'] > 70].sort_values('meantemp', ascending=False).head(5)
top5_hot_humid[['date', 'meantemp', 'humidity']]


,date,meantemp,humidity
48,2017-02-18,21.125000,70.750000
5,2017-01-06,19.318182,79.318182
34,2017-02-04,18.700000,77.600000
3,2017-01-04,18.700000,70.050000
35,2017-02-05,18.631579,77.631579


In [12]:
print("Inference: The single hottest day overall is", top8_hottest.iloc[0]['date'].date(),
      "at", round(top8_hottest.iloc[0]['meantemp'],1), "°C. Once we restrict to",
      "humidity > 70%, the hottest matching day is", top5_hot_humid.iloc[0]['date'].date(),
      "-> hot+humid days cluster later in the recorded period (closer to April),",
      "consistent with Delhi's pre-monsoon warm-up.")


Inference: The single hottest day overall is 2017-04-20 at 34.5 °C. Once we restrict to humidity > 70%, the hottest matching day is 2017-02-18 -> hot+humid days cluster later in the recorded period (closer to April), consistent with Delhi's pre-monsoon warm-up.


## 6. Frequency listing of a column (2 cases)

In [13]:
# Add a 'month' column first (used for frequency + later sections)
df['month'] = df['date'].dt.month_name()

# Case 1: frequency of each month in the dataset
month_freq = df['month'].value_counts()
print(month_freq)


month
January     31
March       31
February    28
April       24
Name: count, dtype: int64


In [14]:
# Case 2: frequency listing after binning meantemp into categories
df['temp_category'] = pd.cut(df['meantemp'], bins=[0, 15, 25, 40],
                              labels=['Cold', 'Mild', 'Hot'])
temp_cat_freq = df['temp_category'].value_counts()
print(temp_cat_freq)


temp_category
Mild    67
Hot     34
Cold    13
Name: count, dtype: int64


In [15]:
print("Inference: January has the most records (", month_freq.max(), "days) simply",
      "because the dataset starts mid-way through it. By temperature category,",
      "'", temp_cat_freq.idxmax(), "' days are the most frequent (",
      temp_cat_freq.max(), "days), showing the dataset is dominated by mild",
      "winter/spring temperatures rather than extreme heat.")


Inference: January has the most records ( 31 days) simply because the dataset starts mid-way through it. By temperature category, ' Mild ' days are the most frequent ( 67 days), showing the dataset is dominated by mild winter/spring temperatures rather than extreme heat.


## 7. Sorting rows and columns — implicit vs. explicit indexing

In [16]:
df_indexed = df.set_index('date')  # date becomes the explicit (label) index

# Explicit indexing (.loc) — by label
explicit_row = df_indexed.loc['2017-02-14']
print("Explicit (.loc by date label):\n", explicit_row)


Explicit (.loc by date label):
 meantemp           16.875
humidity              NaN
wind_speed         6.9625
meanpressure     1021.375
month            February
temp_category        Mild
Name: 2017-02-14 00:00:00, dtype: object


In [17]:
# Implicit indexing (.iloc) — by integer position
implicit_row = df_indexed.iloc[5]
print("Implicit (.iloc by position 5):\n", implicit_row)


Implicit (.iloc by position 5):
 meantemp           19.318182
humidity           79.318182
wind_speed          8.681818
meanpressure     1011.772727
month                January
temp_category           Mild
Name: 2017-01-06 00:00:00, dtype: object


In [18]:
# Sort rows by index (date) and columns alphabetically
sorted_by_index = df_indexed.sort_index()               # sort rows (explicit index)
sorted_columns = df_indexed.sort_index(axis=1)           # sort columns alphabetically
print(sorted_columns.columns.tolist())


['humidity', 'meanpressure', 'meantemp', 'month', 'temp_category', 'wind_speed']


In [19]:
print("Inference: .loc['2017-02-14'] retrieves the row by its date *label*",
      "(explicit indexing), while .iloc[5] retrieves the 6th row purely by its",
      "*position* regardless of the index labels (implicit indexing) - both point",
      "to different rows here, confirming the two indexing modes are independent.")


Inference: .loc['2017-02-14'] retrieves the row by its date *label* (explicit indexing), while .iloc[5] retrieves the 6th row purely by its *position* regardless of the index labels (implicit indexing) - both point to different rows here, confirming the two indexing modes are independent.


## 8. Accessing rows with compound conditions (3 cases), selected columns only

In [20]:
# Case 1: AND condition — hot AND dry days
case1 = df[(df['meantemp'] > 30) & (df['humidity'] < 50)][['date', 'meantemp', 'humidity']]
case1


,date,meantemp,humidity
87,2017-03-29,31.000000,34.500000
89,2017-03-31,30.625000,37.625000
90,2017-04-01,31.375000,35.125000
92,2017-04-03,30.500000,29.750000
95,2017-04-06,31.222222,26.000000
103,2017-04-14,30.500000,37.625000
104,2017-04-15,31.222222,30.444444
105,2017-04-16,31.000000,34.250000
106,2017-04-17,32.555556,38.444444
107,2017-04-18,34.000000,27.333333


In [21]:
# Case 2: OR condition — windy OR low-pressure days
case2 = df[(df['wind_speed'] > 10) | (df['meanpressure'] < 1000)][['date', 'wind_speed', 'meanpressure']]
case2.head()


,date,wind_speed,meanpressure
0,2017-01-01,2.743478,59.000000
6,2017-01-07,10.041667,1011.375000
15,2017-01-16,10.380000,1017.150000
18,2017-01-19,10.338095,1022.809524
19,2017-01-20,11.226316,1021.789474


In [22]:
# Case 3: AND + membership condition — Hot days in a specific set of months
case3 = df[(df['temp_category'] == 'Hot') & (df['month'].isin(['March', 'April']))][['date', 'meantemp', 'month']]
case3


,date,meantemp,month
80,2017-03-22,27.250000,March
81,2017-03-23,28.000000,March
82,2017-03-24,28.916667,March
83,2017-03-25,26.500000,March
84,2017-03-26,29.100000,March
85,2017-03-27,29.500000,March
86,2017-03-28,29.888889,March
87,2017-03-29,31.000000,March
88,2017-03-30,29.285714,March
89,2017-03-31,30.625000,March


In [23]:
print("Inference: Only", len(case1), "days are both hot (>30°C) and dry (<50% humidity) —",
      "a rare combination early in the year. In contrast,", len(case2),
      "days are windy or low-pressure, a much more common pattern. All",
      len(case3), "'Hot' days fall in March/April, confirming the warming trend",
      "toward the end of the recorded period.")


Inference: Only 16 days are both hot (>30°C) and dry (<50% humidity) — a rare combination early in the year. In contrast, 34 days are windy or low-pressure, a much more common pattern. All 34 'Hot' days fall in March/April, confirming the warming trend toward the end of the recorded period.


## 9. Minimum and maximum value analysis

In [24]:
print("Column-wise minimums:\n", df[['meantemp','humidity','wind_speed','meanpressure']].min())
print("\nColumn-wise maximums:\n", df[['meantemp','humidity','wind_speed','meanpressure']].max())


Column-wise minimums:
 meantemp        11.0000
humidity        17.7500
wind_speed       1.3875
meanpressure    59.0000
dtype: float64

Column-wise maximums:
 meantemp          34.500000
humidity          95.833333
wind_speed        19.314286
meanpressure    1022.809524
dtype: float64


In [25]:
hottest_day = df.loc[df['meantemp'].idxmax(), ['date', 'meantemp']]
coldest_day = df.loc[df['meantemp'].idxmin(), ['date', 'meantemp']]
print("Hottest day:\n", hottest_day)
print("\nColdest day:\n", coldest_day)


Hottest day:
 date        2017-04-20 00:00:00
meantemp                   34.5
Name: 109, dtype: object

Coldest day:
 date        2017-01-11 00:00:00
meantemp                   11.0
Name: 10, dtype: object


In [26]:
print("Inference: The temperature swing across the dataset is",
      round(df['meantemp'].max() - df['meantemp'].min(), 1),
      "°C, from a coldest reading of", round(df['meantemp'].min(),1), "°C on",
      coldest_day['date'].date(), "to a hottest reading of", round(df['meantemp'].max(),1),
      "°C on", hottest_day['date'].date(), "— a wide range for a ~4-month window,",
      "reflecting the fast winter-to-spring warm-up in Delhi.")


Inference: The temperature swing across the dataset is 23.5 °C, from a coldest reading of 11.0 °C on 2017-01-11 to a hottest reading of 34.5 °C on 2017-04-20 — a wide range for a ~4-month window, reflecting the fast winter-to-spring warm-up in Delhi.


## 10. GroupBy on one or more columns (2 cases)

In [27]:
# Case 1: group by a single column (month)
by_month = df.groupby('month')['meantemp'].mean().sort_values(ascending=False)
print(by_month)


month
April       30.753663
March       23.753760
February    18.349981
January     15.710873
Name: meantemp, dtype: float64


In [28]:
# Case 2: group by multiple columns (month + temp_category)
by_month_cat = df.groupby(['month', 'temp_category'])['humidity'].mean()
print(by_month_cat)


month     temp_category
April     Hot              30.344772
February  Cold             71.777778
          Mild             64.249705
January   Cold             77.979962
          Mild             76.688230
March     Mild             51.958811
          Hot              39.716548
Name: humidity, dtype: float64


In [29]:
print("Inference: Average meantemp rises steadily from January (coolest,",
      round(by_month.min(),1), "°C) toward April (warmest,", round(by_month.max(),1),
      "°C). Splitting further by temp_category shows 'Hot' days consistently have",
      "lower average humidity than 'Mild'/'Cold' days in the same month,",
      "reinforcing the inverse temp-humidity relationship seen earlier.")


Inference: Average meantemp rises steadily from January (coolest, 15.7 °C) toward April (warmest, 30.8 °C). Splitting further by temp_category shows 'Hot' days consistently have lower average humidity than 'Mild'/'Cold' days in the same month, reinforcing the inverse temp-humidity relationship seen earlier.


## 11. Add a new column derived from existing columns

In [30]:
# Heat Index approximation (simplified) combining temperature and humidity
df['heat_index'] = df['meantemp'] + 0.05 * df['humidity']
df[['date', 'meantemp', 'humidity', 'heat_index']].head()


,date,meantemp,humidity,heat_index
0,2017-01-01,15.913043,85.869565,20.206522
1,2017-01-02,18.500000,77.222222,22.361111
2,2017-01-03,17.111111,81.888889,21.205556
3,2017-01-04,18.700000,70.050000,22.202500
4,2017-01-05,18.388889,74.944444,22.136111


In [31]:
print("Inference: 'heat_index' (meantemp adjusted upward for humidity) peaks at",
      round(df['heat_index'].max(), 1), "on average slightly above raw meantemp,",
      "giving a more realistic 'feels-like' figure that would matter more for a",
      "heatwave early-warning system than raw temperature alone.")


Inference: 'heat_index' (meantemp adjusted upward for humidity) peaks at 36.2 on average slightly above raw meantemp, giving a more realistic 'feels-like' figure that would matter more for a heatwave early-warning system than raw temperature alone.


## 12. Aggregate functions with GroupBy (2 cases)

In [32]:
# Case 1: multiple aggregates on one column, grouped by month
agg1 = df.groupby('month')['meantemp'].agg(['mean', 'min', 'max', 'std']).round(2)
agg1


,mean,min,max,std
month,,,,
April,30.75,25.62,34.50,2.38
February,18.35,14.67,23.38,2.60
January,15.71,11.00,21.00,2.45
March,23.75,17.38,31.00,4.30


In [33]:
# Case 2: different aggregates on different columns, grouped by temp_category
agg2 = df.groupby('temp_category').agg(
    avg_humidity=('humidity', 'mean'),
    max_wind=('wind_speed', 'max'),
    avg_pressure=('meanpressure', 'mean')
).round(2)
agg2


,avg_humidity,max_wind,avg_pressure
temp_category,,,
Cold,77.50,10.38,1017.64
Mild,63.91,16.66,1000.47
Hot,33.18,19.31,1005.86


In [34]:
print("Inference: April shows the highest month-to-month temperature variability",
      "(std =", agg1.loc['April','std'] if 'April' in agg1.index else 'N/A',
      "), consistent with a transition month. 'Hot' days average the lowest",
      "pressure of the three categories, matching the well-known inverse",
      "temperature-pressure relationship.")


Inference: April shows the highest month-to-month temperature variability (std = 2.38 ), consistent with a transition month. 'Hot' days average the lowest pressure of the three categories, matching the well-known inverse temperature-pressure relationship.


## 13. Selecting a particular group by name / condition

In [35]:
# By name: get all rows for the group 'April' directly
april_group = df.groupby('month').get_group('April')
print("April records:", len(april_group))
april_group[['date','meantemp']].head()


April records:

 24


,date,meantemp
90,2017-04-01,31.375000
91,2017-04-02,29.750000
92,2017-04-03,30.500000
93,2017-04-04,30.933333
94,2017-04-05,29.230769


In [36]:
# By condition: keep only groups (months) whose average meantemp exceeds 20°C
warm_months = df.groupby('month').filter(lambda g: g['meantemp'].mean() > 20)
print("Months kept (avg temp > 20°C):", warm_months['month'].unique())


Months kept (avg temp > 20°C): <StringArray>
['March', 'April']
Length: 2, dtype: str


In [37]:
print("Inference: Selecting the 'April' group directly confirms it has",
      len(april_group), "recorded days. The condition-based filter keeps only",
      warm_months['month'].nunique(), "month(s) whose average temperature exceeds",
      "20°C - useful for isolating the 'warm season' subset relevant to heatwave analysis.")


Inference: Selecting the 'April' group directly confirms it has 24 recorded days. The condition-based filter keeps only 2 month(s) whose average temperature exceeds 20°C - useful for isolating the 'warm season' subset relevant to heatwave analysis.


## 14. Correlation between two columns

In [38]:
corr_matrix = df[['meantemp','humidity','wind_speed','meanpressure']].corr()
corr_matrix


,meantemp,humidity,wind_speed,meanpressure
meantemp,1.000000,-0.855604,0.217743,0.030682
humidity,-0.855604,1.000000,-0.328846,-0.099259
wind_speed,0.217743,-0.328846,1.000000,0.130352
meanpressure,0.030682,-0.099259,0.130352,1.000000


In [39]:
r = df['meantemp'].corr(df['humidity'])
print("Correlation(meantemp, humidity):", round(r, 3))


Correlation(meantemp, humidity): -0.856


In [40]:
print("Inference: meantemp and humidity have a strong negative correlation (r =",
      round(r,2), "), meaning warmer days in this dataset are reliably drier —",
      "one of the clearest relationships in the data, and a useful predictor",
      "signal for a temperature-forecasting model.")


Inference: meantemp and humidity have a strong negative correlation (r = -0.86 ), meaning warmer days in this dataset are reliably drier — one of the clearest relationships in the data, and a useful predictor signal for a temperature-forecasting model.


## 15. Transformation — Normalization

In [41]:
# Min-Max normalization (scales to [0, 1])
df['meantemp_minmax'] = (df['meantemp'] - df['meantemp'].min()) / (df['meantemp'].max() - df['meantemp'].min())

# Z-score standardization (mean 0, std 1)
df['meantemp_zscore'] = (df['meantemp'] - df['meantemp'].mean()) / df['meantemp'].std()

df[['date', 'meantemp', 'meantemp_minmax', 'meantemp_zscore']].head()


,date,meantemp,meantemp_minmax,meantemp_zscore
0,2017-01-01,15.913043,0.209066,-0.911945
1,2017-01-02,18.500000,0.319149,-0.505195
2,2017-01-03,17.111111,0.260047,-0.723572
3,2017-01-04,18.700000,0.327660,-0.473749
4,2017-01-05,18.388889,0.314421,-0.522665


In [42]:
print("Inference: After Min-Max scaling, meantemp values now range from",
      round(df['meantemp_minmax'].min(),2), "to", round(df['meantemp_minmax'].max(),2),
      "(as expected for [0,1] scaling). The Z-score column shows how many standard",
      "deviations each day's temperature is from the mean — useful for spotting",
      "anomalous days (|z| > 2) that could indicate an early heatwave signal.")


Inference: After Min-Max scaling, meantemp values now range from

 0.0 to 1.0 (as expected for [0,1] scaling). The Z-score column shows how many standard deviations each day's temperature is from the mean — useful for spotting anomalous days (|z| > 2) that could indicate an early heatwave signal.


## 16. Joining, Merging, and Concatenation

We build a small **season lookup table** and merge it onto the main dataframe by
month, then demonstrate **concatenation** by splitting the data in half and
stitching it back together.

In [43]:
# --- MERGE: join a small lookup dataframe onto df by 'month' ---
season_lookup = pd.DataFrame({
    'month': ['January','February','March','April'],
    'season': ['Winter','Winter','Spring','Spring']
})

df_merged = pd.merge(df, season_lookup, on='month', how='left')
df_merged[['date','month','season','meantemp']].head()


,date,month,season,meantemp
0,2017-01-01,January,Winter,15.913043
1,2017-01-02,January,Winter,18.500000
2,2017-01-03,January,Winter,17.111111
3,2017-01-04,January,Winter,18.700000
4,2017-01-05,January,Winter,18.388889


In [44]:
# --- CONCAT: split into two halves by date, then concatenate back together ---
first_half = df.iloc[:57]
second_half = df.iloc[57:]
df_concat = pd.concat([first_half, second_half], axis=0)

print("first_half rows:", len(first_half), "| second_half rows:", len(second_half))
print("Concatenated back shape:", df_concat.shape, "| matches original:", df_concat.shape == df.shape)


first_half rows: 57 | second_half rows: 57
Concatenated back shape: (114, 10) | matches original: True


In [45]:
print("Inference: The merge successfully tagged every row with its 'season'",
      "(Winter/Spring) using 'month' as the join key, with no unmatched rows",
      "(how='left' preserved all", len(df_merged), "records). The concat step",
      "confirms splitting the data and stitching it back together reproduces the",
      "original", df.shape[0], "-row dataset exactly.")


Inference: The merge successfully tagged every row with its 'season' (Winter/Spring) using 'month' as the join key, with no unmatched rows (how='left' preserved all 114 records). The concat step confirms splitting the data and stitching it back together reproduces the original 114 -row dataset exactly.
